# Is your hometown getting hotter?

**Terrain beginner · Python project · NOAA Climate Data**

Download daily temperature records for a U.S. weather station, compute annual averages, and estimate a long-run warming trend.

---

## Learning objectives

By the end, you should be able to:

- Download daily climate data from a public API
- Resample daily values to annual averages
- Fit and interpret a linear trend (warming rate per decade)
- Explain limits of a single-station time series

**Time:** ~1 hr · **Dataset:** [NOAA Climate Data explainer](../explainer.html?id=noaa-climate)

---

## For mentors

- **Prerequisites:** Beginner air-quality project helps but is not required.
- **Personal hook:** Have the student change `STATION_ID` to their hometown before Section 2 (find IDs at [NOAA Station Search](https://www.ncei.noaa.gov/cdo-web/)).
- **Watch for:** NOAA stores TMAX in tenths of °C — we convert in code; explain why numbers look like 750 before conversion.
- **Live data:** Default pulls from NOAA with no API token (uses NCEI daily-summaries service).

**Suggested flow:** Read each section aloud briefly → student runs the cell → use the *Mentor check-in* question before moving on. Do not skip the final takeaway.

---


**Goal:** Load libraries and set your study location.

**Concept:** We analyze **daily maximum temperature (TMAX)** — a common choice for heat trends. `STATION_ID` identifies one NOAA weather station.

**Run the cell below**, then compare your output to what we expect.

**You should see:** No output. Confirm `CITY_LABEL` matches the station you chose.

**Mentor check-in:** Ask: *Why one station instead of a whole city average?* (Trade-off: simpler data, less representative.)


In [ ]:
from io import StringIO
from pathlib import Path
import pandas as pd
import requests
import matplotlib.pyplot as plt
import numpy as np

STATION_ID = "USW00023174"   # LAX — change to your hometown station
CITY_LABEL = "Los Angeles (LAX)"
START_DATE = "1980-01-01"
END_DATE = "2024-12-31"
USE_BUNDLED_SAMPLE = False   # set True only if offline

**Goal:** Download daily TMAX from NOAA.

**Concept:** Each row is one day's maximum temperature at the station. Thousands of rows = normal.

**Run the cell below**, then compare your output to what we expect.

**You should see:** Message with row count (often 10,000+) and a preview with `STATION`, `DATE`, `TMAX`.

**Mentor check-in:** Ask: *What unit do you think TMAX is in before we convert?*

**If something breaks:** Timeout? Narrow the date range. Offline? Set `USE_BUNDLED_SAMPLE = True`.


In [ ]:
def fetch_noaa_daily(station, start, end):
    url = "https://www.ncei.noaa.gov/access/services/data/v1"
    params = {
        "dataset": "daily-summaries",
        "stations": station,
        "startDate": start,
        "endDate": end,
        "dataTypes": "TMAX",
        "format": "csv",
    }
    resp = requests.get(url, params=params, timeout=120)
    resp.raise_for_status()
    return pd.read_csv(StringIO(resp.text))

def load_noaa_sample():
    candidates = [Path("data/noaa_lax_tmax_daily.csv"), Path("notebooks/data/noaa_lax_tmax_daily.csv")]
    for path in candidates:
        if path.exists():
            return pd.read_csv(path), str(path)
    import urllib.request
    url = "https://raw.githubusercontent.com/KrishM23/Environmental-Data-Science/main/notebooks/data/noaa_lax_tmax_daily.csv"
    with urllib.request.urlopen(url, timeout=60) as resp:
        return pd.read_csv(StringIO(resp.read().decode())), url

if USE_BUNDLED_SAMPLE:
    daily, source = load_noaa_sample()
    print(f"Using bundled sample from {source}")
else:
    daily = fetch_noaa_daily(STATION_ID, START_DATE, END_DATE)
    print(f"Downloaded {len(daily):,} daily rows for {STATION_ID}")

daily.head()

**Goal:** Convert units and compute annual means.

**Concept:** NOAA archives TMAX in **tenths of °C** (e.g. 250 → 25.0 °C). We convert to °F because many U.S. students think in Fahrenheit, then average by year.

**Run the cell below**, then compare your output to what we expect.

**You should see:** A small table: one row per year with `mean_tmax_f`.

**Mentor check-in:** Ask: *Why average by year instead of plotting every day?* (Reduces noise; shows long-run signal.)


In [ ]:
work = daily.copy()
work["DATE"] = pd.to_datetime(work["DATE"])
work["TMAX_C"] = pd.to_numeric(work["TMAX"], errors="coerce") / 10.0
work = work.dropna(subset=["TMAX_C"])
work["TMAX_F"] = work["TMAX_C"] * 9 / 5 + 32

annual = (
    work.groupby(work["DATE"].dt.year, as_index=False)["TMAX_F"]
        .mean()
        .rename(columns={"DATE": "year", "TMAX_F": "mean_tmax_f"})
)
print(f"Years covered: {annual['year'].min()}–{annual['year'].max()} ({len(annual)} years)")
annual.tail()

**Goal:** Quantify the warming trend.

**Concept:** We fit a straight line through annual means. Slope × 10 ≈ **°F per decade** — a headline number students can cite.

**Run the cell below**, then compare your output to what we expect.

**You should see:** Printed mean temperature and trend in °F/decade (positive = warming).

**Mentor check-in:** Ask: *Is the slope positive? Does that match what you expected for this place?*


In [ ]:
x = annual["year"].to_numpy()
y = annual["mean_tmax_f"].to_numpy()
slope, intercept = np.polyfit(x, y, 1)
warming_per_decade = slope * 10

print(f"Mean annual TMAX: {y.mean():.1f} °F")
print(f"Trend: {warming_per_decade:+.2f} °F per decade ({START_DATE[:4]}–{END_DATE[:4]})")

**Goal:** Plot the story.

**Concept:** Dots = observed annual means. Line = linear trend. Walk through one hot year and one cool year on the chart.

**Run the cell below**, then compare your output to what we expect.

**You should see:** Scatter + trend line with legend showing °F/decade.

**Mentor check-in:** Ask: *Would you call this a perfect straight line in real life?* (No — natural variability remains.)


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.scatter(annual["year"], annual["mean_tmax_f"], s=18, alpha=0.75, color="#5B8BB0", label="Annual mean TMAX")
trend = intercept + slope * x
ax.plot(x, trend, color="#C45C5C", linewidth=2,
        label=f"Trend: {warming_per_decade:+.2f} °F/decade")
ax.set_title(f"Is {CITY_LABEL} getting hotter?")
ax.set_xlabel("Year")
ax.set_ylabel("Mean daily maximum temp (°F)")
ax.legend()
plt.tight_layout()
plt.show()

---

## Final takeaway

Student writes 3–4 sentences. **Mentor check-in prompts:**

1. Is the trend positive? By how much per decade?
2. Have them cite **station ID** and **years of data** in one sentence (methods practice).
3. Why is one weather station ≠ an entire city's climate?

**Extension:** Repeat with a second station (inland vs. coastal) and compare slopes.

**Next:** [NOAA explainer](../explainer.html?id=noaa-climate) · [Air quality map](la_air_quality_map.ipynb)
